# Multi-Garment Detection — RF-DETR-Seg-Small on Fashionpedia

End-to-end training + Core ML export notebook. Produces
`RFDETRSegFashion.mlpackage` ready to drop into
`WardrobeReDo/Models/CoreML/`.

**Status (2026-04-18):** scaffold. The recipe below is the agreed-on
plan from
[`docs/plans/2026-04-18-multi-garment-detection.md`](../../docs/plans/2026-04-18-multi-garment-detection.md)
Sections 2-5. The GPU run that produces the shipped `.mlpackage` has
not yet been executed.

**Budget:** ~30-50 A100 GPU hours; ~$100-200 on Lambda / Vast.ai.

**Outputs:** `checkpoints/best.pth` (PyTorch) and
`WardrobeReDo/Models/CoreML/RFDETRSegFashion.mlpackage` (Core ML,
6-bit palettized, ~30-50 MB).

## 0. Environment sanity check

Run these first — mismatched versions are the #1 source of
`coremltools.convert(...)` failures with DETR-family models.

In [ ]:
import sys
import torch
import coremltools as ct

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:  {torch.cuda.get_device_name(0)}")
print(f"coremltools:  {ct.__version__}")

# Pinned in requirements.txt — bail loudly if the notebook is running
# against a drifted environment.
assert torch.__version__.startswith("2.5"),    "torch 2.5.x expected"
assert ct.__version__.startswith("8.1"),        "coremltools 8.1.x expected"

## 1. Download Fashionpedia

The HuggingFace `detection-datasets/fashionpedia` mirror is the most
convenient host — it auto-resolves the CC-BY-4.0 annotations +
CC-licensed image subset. Total: ~47k images, ~4 GB on disk after
decompression.

License reminder: annotations are CC BY 4.0; attribution is required.
Our app's About screen already carries the credit line — see plan
Section 4 'Data licensing checklist'.

In [ ]:
from pathlib import Path
from datasets import load_dataset

DATA_ROOT = Path("./data/fashionpedia")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

ds = load_dataset("detection-datasets/fashionpedia", cache_dir=str(DATA_ROOT))
print(ds)
print("Train size:", len(ds["train"]))
print("Val size:  ", len(ds["val"]))
print("Sample:", {k: type(v).__name__ for k, v in ds["train"][0].items()})

## 2. Class collapse: Fashionpedia → ClothingCategory

Fashionpedia has 27 main apparel classes. The iOS `ClothingCategory`
enum has 6 cases. The mapping lives in the Swift side
(`WardrobeReDo/Models/Enums/ClothingCategory.swift` —
`fromFashionpediaClass(_:)`), BUT we keep the raw 27-way labels in
`MaskProposal.modelClassRaw` so v1.1 can split `.accessory` without
retraining the model.

Therefore: **train on raw Fashionpedia labels, not the collapsed 6.**
The collapse happens at inference time in Swift.

In [ ]:
# Same label order the Swift `fromFashionpediaClass(_:)` expects.
# Keep in sync with the enum — regenerate both from this single list.
FASHIONPEDIA_CLASSES = [
    "shirt_blouse", "top_t-shirt_sweatshirt", "sweater", "cardigan",
    "jacket", "vest", "coat", "cape",
    "pants", "shorts", "skirt", "tights_stockings",
    "dress", "jumpsuit",
    "shoe", "boot", "sandal", "sock", "leg_warmer",
    "glasses", "hat", "headband", "scarf", "tie",
    "bag_wallet", "belt",
    "glove", "watch", "ring", "bracelet", "earring", "necklace",
    "umbrella",
]
NUM_CLASSES = len(FASHIONPEDIA_CLASSES)
print(f"Training {NUM_CLASSES} classes")

## 3. Load RF-DETR-Seg-Small (COCO-pretrained)

Starting from the COCO checkpoint is essential — training from scratch
on Fashionpedia alone overfits. Roboflow's checkpoint is Apache 2.0.

In [ ]:
from rfdetr import RFDETRSegSmall

model = RFDETRSegSmall(pretrained=True, num_classes=NUM_CLASSES)
print(model)

## 4. Training

- Epochs: 6-12 (DETR fine-tunes in fewer epochs than CNNs)
- LR: 1e-4 → 5e-5 cosine
- Batch: 4-8 per GPU at 1024×1024, mixed precision
- Wall clock: ~30-50 GPU hours on 1× A100
- Target: ≥30 mask mAP @ 0.5 IoU on the 6 collapsed superclasses,
  ≥22 mAP on the 27 raw classes

Use early-stop on val mAP plateau. Save epoch snapshots in case a
later epoch overfits.

In [ ]:
# Placeholder. The real training loop is provided by rfdetr's Trainer;
# this is where you plug in the dataset + output directory. See
# https://github.com/roboflow/rf-detr#training for the current API.

# model.train(
#     dataset="./data/fashionpedia",
#     epochs=10,
#     batch_size=4,
#     image_size=1024,
#     lr=1e-4,
#     output_dir="./checkpoints",
# )

## 5. Validate on the Fashionpedia val split

- Per-class mAP to surface weak classes (accessories tend to lag)
- Confusion matrix over the 27 raw classes
- Sample predictions exported to `./samples/*.png` for visual inspection

In [ ]:
# metrics = model.evaluate(dataset="./data/fashionpedia", split="val")
# print(metrics)

## 6. Export to Core ML

Direct PyTorch → Core ML MIL. **Not** through ONNX — coremltools 8+
handles DETR-style attention ops directly, and ONNX adds an
opportunistic reshape that costs ANE residency.

In [ ]:
import torch
import coremltools as ct

# 1. Load the best checkpoint.
# model.load_state_dict(torch.load("./checkpoints/best.pth"))
model.eval()

# 2. Trace with a representative input (fixed shape — critical for ANE).
example_input = torch.rand(1, 3, 1024, 1024)
traced = torch.jit.trace(model, example_input)

# 3. Convert to Core ML ML Program.
mlmodel = ct.convert(
    traced,
    inputs=[
        ct.ImageType(
            name="image",
            shape=(1, 3, 1024, 1024),
            scale=1 / 255.0,
            bias=[0, 0, 0],
        )
    ],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS17,
    compute_units=ct.ComputeUnit.ALL,
)

mlmodel.save("RFDETRSegFashion_fp16.mlpackage")

## 7. 6-bit palettization

FP16 baseline is ~125 MB. Six-bit palettization gets us to ~30-50 MB
— the same compression SAM2Extractor uses, so latency characteristics
are predictable.

Skip 4-bit; visual regression on mask edges is real at 4-bit even
though the numeric drop in mAP is small.

In [ ]:
from coremltools.optimize.coreml import (
    OpPalettizerConfig,
    OptimizationConfig,
    palettize_weights,
)

palettize_config = OpPalettizerConfig(
    nbits=6,
    mode="kmeans",
    granularity="per_grouped_channel",
    group_size=16,
)
config = OptimizationConfig(global_config=palettize_config)

compressed = palettize_weights(mlmodel, config=config)
compressed.save("RFDETRSegFashion.mlpackage")

# Move the compressed model into the iOS app's bundle directory.
# xcodegen already references this path via the Resources phase —
# no project edit required.
import shutil
shutil.copytree(
    "RFDETRSegFashion.mlpackage",
    "../../WardrobeReDo/Models/CoreML/RFDETRSegFashion.mlpackage",
    dirs_exist_ok=True,
)

## 8. Post-export checklist

Before flipping `FeatureFlags.isMultiGarmentEnabled` to `true` by
default (Commit 9 of the canonical plan), verify:

- [ ] `xcodegen generate` picks up the new model file
- [ ] App builds and runs the DEBUG smoke test without throwing
  (`Settings → Developer → ML Diagnostics → Status: passed`)
- [ ] Inference completes under 1.5 s on iPhone 14 (Instruments →
  Core ML template)
- [ ] Most ops show on the ANE compute lane (residency check)
- [ ] Manual test on the user's reported photo (4-garment case)
  surfaces 3+ proposals with sensible bounding boxes
- [ ] Update `docs/plans/INDEX.md` status to `shipped`